### Plotting setup

In [6]:
# basic imports
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import glob
import pandas as pd

# ============ for latex fonts ============
from matplotlib import rc  # , font_manager

rc(
    "text.latex", preamble=r"\usepackage{lmodern} \usepackage{physics}"
)  # this helps use the plots in tex files
plt.rcParams.update({"font.size": 11})
plt.rcParams.update(
    {
        "xtick.labelsize": 8,
        "ytick.labelsize": 8,
        "xtick.major.width": 0.8,
        "ytick.major.width": 0.8,
        "xtick.minor.width": 0.8,
        "ytick.minor.width": 0.8,
        "axes.labelsize": 9,
        "axes.titlesize": 10,
        "axes.linewidth": 0.8,
        "lines.linewidth": 0.8,
        "patch.linewidth": 0.8,
        "legend.fontsize": 8,
        "legend.title_fontsize": 9,
        "legend.fancybox": False,
        "legend.frameon": False,
        "legend.handlelength": 1.0,
        # "legend.handletextpad": 0.5,
        "legend.labelspacing": 0.4,
        "figure.titlesize": 12,
        "figure.figsize": [3.54, 3.0],
        "figure.dpi": 300,
        "savefig.dpi": 300,
        "savefig.bbox": "tight",
        "mathtext.fontset": "cm",
        "text.usetex": True,
        "font.family": "Computer Modern Roman",
    }
)
# ==========================================

###################

data_dir = Path("build/data")
plot_dir = Path("plot")

if not Path.exists(plot_dir):
    Path.mkdir(plot_dir)

###################


def load_data(filename, data_dir=data_dir):
    data = np.loadtxt(data_dir / filename)
    return (col for col in data.T)


def prep_image_data(x, y, z):
    x = np.unique(x)
    y = np.unique(y)

    X, Y = np.meshgrid(x, y)
    Z = z.reshape(len(x), len(y))

    return (np.transpose(Z), [min(x), max(x), min(y), max(y)])
    # data = pd.DataFrame({"x": x, "y": y, "z": z})
    # data_pivot = data.pivot_table(index="y", columns="x", values="z")
    # extent = [min(x), max(x), min(y), max(y)]
    # return data_pivot, extent


def set_labels(x=r"$x$", y=r"$y$", title=""):
    plt.xlabel(x)
    plt.ylabel(y)
    plt.title(title)


def save_plot(filename, dpi=300, bbox_inches="tight", plot_dir=plot_dir):
    plt.tight_layout()
    plt.savefig(plot_dir / filename, bbox_inches=bbox_inches, dpi=dpi)
    plt.close("all")


##########################

def symlog10(x):
    return np.sign(x) * np.log10((np.abs(x) + 1.0))

### slice from 2D data

In [5]:
def get_vertical_slice(x, y, z, axis, offset=0.0):
    ux = np.unique(x)
    uy = np.unique(y)

    match axis:
        case "x":
            diff = np.abs(ux - offset)
            val = ux[np.argmin(diff)]

            return uy, z[x == val]

        case "y":
            diff = np.abs(uy - offset)
            val = uy[np.argmin(diff)]

            return ux, z[y == val]

        case _:
            raise ValueError("Invalid axis")


# something

### Band structure slice

In [9]:
k, *energies = load_data(f"BS.dat")
for band_idx, E in enumerate(energies):
    plt.plot(k, E)

# plt.xlim(-0.5, 0.5)
# plt.ylim(-0.5, 0.5)
plt.ylim(-1, 1)
# plt.ylim(-50, 50)
set_labels(r"$k_x\ (1/a)$ ", r"$E$ (meV)", r"$\mu = -3.0$ meV, $k_y = 0.0$")
plt.grid()
save_plot("BS.png")

### Band structure 3D

In [ ]:
kx, ky, *energies = load_data("BS.dat")
ukx, uky = np.unique(kx), np.unique(ky)
x, y = np.meshgrid(ukx, uky)

fig = plt.figure(figsize=(7, 7))
ax = fig.add_subplot(111, projection="3d")

for i, e in enumerate(energies):
    ax.plot_surface(x, y, e.reshape(len(ukx), len(uky)).T, alpha=0.5, label=f"{i}")

set_labels(r"$k_x$", r"$k_y$", r"$E$ (meV)")
ax.legend(title="Band index")
save_plot("bands_3d.png")

### Gap closing wiht changes in mu

In [ ]:
fig, axes = plt.subplots(3, 3, sharex=True, sharey=True, figsize=(5, 5.5))

mu_start, mu_end = -2.5, -2.1

for i, mu in enumerate(np.linspace(mu_start, mu_end, 9)):
    k, *energies = load_data(f"BS{i}.dat")

    for band_idx, E in enumerate(energies):
        axes.flat[i].plot(k, E)

    axes.flat[i].set_title(f"$\mu = {mu:.2f}$ meV")

for ax in axes.flat:
    # ax.set_xlim(-1, 1)
    ax.set_xlim(-0.15, 0.15)

    ax.set_ylim(-0.3, 0.3)
    # ax.set_ylim(-4, 0)
    ax.set_xlabel(r"$k_x$ ($1/a$)")
    ax.set_ylabel(r"$E$ (meV)")
    ax.set_box_aspect(1)
    ax.label_outer()
    ax.grid()

save_plot(f"gap_closing.png")

### Chern numbers vs mu

In [5]:
n_bands = 4
n_dense, n_sparse = 1000, 500

# n_bands = 12
# n_dense, n_sparse = 5000, 1500

mu, *Cs = load_data("mu_Cs.dat")

for i, C in enumerate(Cs[: (n_bands // 2)]):
    plt.plot(mu, C, label=f"{i}")

plt.plot(mu, np.sum(Cs[: (n_bands // 2)], axis=0), label="Sum")

plt.grid(alpha=0.5)
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left", title="Band")
set_labels(r"$\mu$ (meV)", r"$C$", f"{n_dense=}, {n_sparse=}")

save_plot(f"mu_Cs_{n_dense}_{n_sparse}.png")

### Chern numbers vs mu and Bz grid

In [8]:
n_bands = 12

# mu, Bz, *Cs = load_data("mu_Bz_Cs.dat")
mu, Bz, *Cs = np.loadtxt("LAOSTO/mu_Bz_Cs_4000_1000.dat", unpack=True)

C = np.sum(Cs[: (n_bands // 2)], axis=0)

Z, e = prep_image_data(mu, Bz, C)
plt.imshow(Z, extent=e, origin="lower", aspect="auto", cmap="RdBu", vmin=-1, vmax=1)

plt.colorbar()
set_labels(r"$\mu$ (meV)", r"$B_z$ (T)")

save_plot("mu_Bz_Cs.png")



for i, C in enumerate(Cs[: (n_bands // 2)]):
    Z, e = prep_image_data(mu, Bz, C)
    # Z[Z.shape[0] // 2-1].fill(np.nan)
    # Z[Z.shape[0] // 2].fill(np.nan)
    # Z[Z.shape[0] // 2+1].fill(np.nan)
    plt.imshow(Z, extent=e, origin="lower", aspect="auto", cmap="RdBu")
    plt.colorbar()
    set_labels(r"$\mu$ (meV)", r"$B_z$ (T)", f"Band {i}")
    save_plot(f"mu_Bz_C_{i}.png")

### Berry Curvature

In [24]:
kx, ky, *BCs = load_data("BC.dat")
fig, axes = plt.subplots(3, 4, sharex=True, sharey=True, figsize=(10, 7))

for iBC, BC in enumerate(BCs):
    BC = symlog10(BC)
    Z, e = prep_image_data(kx, ky, BC)
    cmax = np.abs(BC).max()
    mapped = axes.flat[iBC].imshow(
        Z, extent=e, origin="lower", aspect="equal", cmap="RdBu", vmin=-cmax, vmax=cmax
    )
    axes.flat[iBC].set_xlabel(r"$k_x$")
    axes.flat[iBC].set_ylabel(r"$k_y$")
    axes.flat[iBC].label_outer()
    axes.flat[iBC].set_title(f"Band {iBC}")
    axes.flat[iBC].set_box_aspect(1)
    fig.colorbar(mapped, ax=axes.flat[iBC])

fig.suptitle(r"SymLog10 of Berry Curvature for $\mu = 0.0$ meV")
save_plot(f"BC.png", dpi=600)

### Gap vs momentum

In [85]:
kx, ky, *gaps = load_data("gap.dat")
gap = gaps[0]

Z, e = prep_image_data(kx, ky, gap)
plt.imshow(Z, extent=e, origin="lower", aspect="equal", cmap="jet")

plt.colorbar()
set_labels(r"$k_x$", r"$k_y$", r"Gap (meV)")

save_plot("gap.png")

### FS contours

In [ ]:
for i, contour_file in enumerate(glob.glob("build/data/FS*.dat")):
    kx, ky = np.loadtxt(contour_file, unpack=True)
    plt.plot(kx, ky, label=f"{i}")

# plt.xlim(-0.3, 0.3)
# plt.ylim(-0.3, 0.3)
plt.gca().set_box_aspect(1)
set_labels(r"$k_x$", r"$k_y$", r"FS")

save_plot("FS.png")

### FS contours and gap along them

In [5]:
for i, contour_file in enumerate(glob.glob("build/data/gap_FS*.dat")):
    kx, ky, *_ = np.loadtxt(contour_file, unpack=True)
    plt.plot(kx, ky, label=f"{i}")

plt.xlim(-0.3, 0.3)
plt.ylim(-0.3, 0.3)
# plt.xlim(-1.2, 1.2)
# plt.ylim(-1.2, 1.2)
plt.gca().set_box_aspect(1)
set_labels(r"$k_x$", r"$k_y$", r"FS")

save_plot("FS.png")

for ib in range(12):
    for i, contour_file in enumerate(glob.glob("build/data/gap_FS*.dat")):
        kx, ky, *gaps = np.loadtxt(contour_file, unpack=True)
        theta = np.arctan2(ky[1:], kx[1:])
        gap = gaps[ib][1:]
        plt.plot(theta, gap, label=f"{i}")

    plt.xlim(-np.pi, np.pi)
    # plt.ylim(0.0, 0.05)
    plt.xticks(
        np.linspace(-np.pi, np.pi, 5),
        [r"$-\pi$", r"$-\pi/2$", r"$0$", r"$\pi/2$", r"$\pi$"],
    )
    # plt.legend(title="contour idx")
    # plt.yscale("log")
    set_labels(r"$\theta$", r"$\Delta$", r"Gap along FS contours")

    save_plot(f"gap_along_FS_{ib}.png")

# #free up variables
# del  gaps, contour_file, kx, ky, theta, gap

### FS countours colored according to gap - long plotting time


In [15]:
kx_ky_gap_lst = []
for i, contour_file in enumerate(glob.glob("build/data/gap_FS*.dat")):
    kx, ky, *gaps = np.loadtxt(contour_file, unpack=True)
    kx_ky_gap_lst.append((kx, ky, gaps[0]))

import matplotlib.cm as cm
import matplotlib.colors as mcolors

all_g = np.concatenate([g for _, _, g in kx_ky_gap_lst])
norm = mcolors.Normalize(vmin=all_g.min(), vmax=all_g.max())
cmap = cm.jet  # Colormap

for x, y, z in kx_ky_gap_lst:
    colors = cmap(norm(z))  # Map z-values to colors
    for i in range(len(x) - 1):
        plt.plot(x[i : i + 2], y[i : i + 2], color=colors[i])

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
plt.colorbar(sm, ax=plt.gca())

set_labels(r"$k_x$", r"$k_y$", r"Gap (meV)")

save_plot("FS_gap_colored.png")

### Abs delta

In [3]:
kx, ky, *abs_deltas = load_data("abs_delta.dat")

lst = []

for i, abs_delta in enumerate(abs_deltas):
    Z,e = prep_image_data(kx, ky, abs_delta)
    plt.imshow(Z, extent=e, origin="lower", aspect="equal", cmap="jet")
    # plt.imshow(np.sqrt(np.abs(Z)), extent=e, origin="lower", aspect="equal", cmap="jet")
    set_labels(r"$k_x$", r"$k_y$", r"$\Delta^2$ (meV)")
    plt.colorbar() 
    save_plot(f"abs_delta_{i}.png")
    
# # plot sum of abs_deltas
# # Z,e = prep_image_data(kx, ky, np.sum(list(map(np.abs, abs_deltas)), axis=0))
# Z,e = prep_image_data(kx, ky, np.sum(abs_deltas, axis=0))
# plt.imshow(Z, extent=e, origin="lower", aspect="equal", cmap="jet")
# set_labels(r"$k_x$", r"$k_y$", r"$\sum\Delta^2$ (meV)")
# plt.colorbar()
# save_plot(f"abs_delta_sum.png")

### Abs delta vs Bz

In [ ]:
Bz, *abs_deltas = load_data("abs_delta_Bz.dat")

for i, abs_delta in enumerate(abs_deltas):
    plt.plot(Bz, abs_delta, label=f"{i}")
    
plt.legend(title="Band")
plt.grid()
set_labels(r"$B_z$ (T)", r"$|\Delta|$ (meV)")
save_plot("abs_delta_Bz.png")

### Delta matrix real and imag

In [17]:
kx, ky, *ds = load_data("delta.dat")

for i in range(len(ds) // 8):  # ith matrix
    fig, axes = plt.subplots(2, 4, sharex=True, sharey=True, figsize=(10, 5))
    d = ds[i * 8 : (i + 1) * 8]
    
    for j in range(8):
        Z, e = prep_image_data(kx, ky, d[j])
        mapped = axes.flat[j].imshow(
            Z, extent=e, origin="lower", aspect="equal", cmap="jet"
        )
        axes.flat[j].set_xlabel(r"$k_x$")
        axes.flat[j].set_ylabel(r"$k_y$")
        axes.flat[j].label_outer()
        axes.flat[j].set_box_aspect(1)
        fig.colorbar(mapped, ax=axes.flat[j])
    
        # for w, contour_file in enumerate(glob.glob("build/data/gap_FS*.dat")):
        #     kxg, kyg, *_ = np.loadtxt(contour_file, unpack=True)
        #     axes.flat[j].plot(kxg, kyg, label=f"{w}")

    # fig.suptitle(r"Real and Imaginary parts of $\Delta$ for band")
    save_plot(f"delta_{i}.png", dpi=600)
    

### Delta matrix phases (from re and im - may be not exact)

In [18]:
kx, ky, *ds = load_data("delta.dat")

for i in range(len(ds) // 8):  # ith matrix
    fig, axes = plt.subplots(2, 2, sharex=True, sharey=True, figsize=(5, 5))
    d = ds[i * 8 : (i + 1) * 8]
    for k in range(0, 8, 2):
        Z, e = prep_image_data(kx, ky, np.arctan2(d[k + 1], d[k]))
        mapped = axes.flat[k // 2].imshow(
            Z, extent=e, origin="lower", aspect="equal", cmap="jet"
        )
        axes.flat[k // 2].set_xlabel(r"$k_x$")
        axes.flat[k // 2].set_ylabel(r"$k_y$")
        axes.flat[k // 2].label_outer()
        axes.flat[k // 2].set_box_aspect(1)
        fig.colorbar(mapped, ax=axes.flat[k // 2])
    
    # fig.suptitle(r"Real and Imaginary parts of $\Delta$ for band")
    save_plot(f"delta_phase_{i}.png", dpi=600)

### Delta matrix determinants and traces

In [19]:
kx, ky, *DTs = load_data("DT.dat")

for i in range(len(DTs) // 4):  # ith matrix
    fig, axes = plt.subplots(2, 2, sharex=True, sharey=True, figsize=(5, 5))
    DT = DTs[i * 4 : (i + 1) * 4]
    for k in range(4):
        Z, e = prep_image_data(kx, ky, DT[k])
        mapped = axes.flat[k].imshow(
            Z, extent=e, origin="lower", aspect="equal", cmap="jet"
        )
        axes.flat[k].set_xlabel(r"$k_x$")
        axes.flat[k].set_ylabel(r"$k_y$")
        axes.flat[k].label_outer()
        axes.flat[k].set_box_aspect(1)
        fig.colorbar(mapped, ax=axes.flat[k])
    
    # fig.suptitle(r"Real and Imaginary parts of $\Delta$ for band")
    save_plot(f"DT_{i}.png", dpi=600)

### Delta matrix determinants and traces, abs and phase

In [27]:
kx, ky, *DTs = load_data("DT.dat")

for i in range(len(DTs) // 4):  # ith matrix
    fig, axes = plt.subplots(2, 2, sharex=True, sharey=True, figsize=(5, 5))
    DT = DTs[i * 4 : (i + 1) * 4]
    for k in range(4):
        if k % 2 == 0:
            Z, e = prep_image_data(kx, ky, np.sqrt(DT[k]**2+DT[k+1]**2))
        else:
            Z, e = prep_image_data(kx, ky, np.arctan2(DT[k], DT[k-1]))

        mapped = axes.flat[k].imshow(
            Z, extent=e, origin="lower", aspect="equal", cmap="jet"
        )
        axes.flat[k].set_xlabel(r"$k_x$")
        axes.flat[k].set_ylabel(r"$k_y$")
        axes.flat[k].label_outer()
        axes.flat[k].set_box_aspect(1)
        fig.colorbar(mapped, ax=axes.flat[k])
        
    
    # fig.suptitle(r"Real and Imaginary parts of $\Delta$ for band")
    save_plot(f"DT_abs_phase_{i}.png", dpi=600)


### Gap and contour interpolation - bad results


In [21]:

# import scipy as scp
# interp = scp.interpolate.RegularGridInterpolator((kxu, kyu), np.reshape(gap, (len(kxu), len(kyu))))

# #for plotting gap along contour
# import matplotlib.cm as cm
# import matplotlib.colors as mcolors

# xyzs = []

# for i,e in enumerate(energies):
#     co=plt.contour(kxe, kye, e.reshape(len(kxe), len(kye)), levels=[0], colors="black", linewidths=0.5)

#     # retrive gap values along the contour
#     for poly in co.allsegs[0]:
#         if len(poly) < 3:
#             continue

#         x = poly[:,0]
#         y = poly[:,1]
#         z=interp(poly)

#         xyzs.append((x,y,z))

# plt.close("all")

# all_z = np.concatenate([z for _,_,z in xyzs])
# norm = mcolors.Normalize(vmin=all_z.min(), vmax=all_z.max())
# cmap = cm.jet  # Colormap

# for x,y,z in xyzs:
#     colors = cmap(norm(z))  # Map z-values to colors
#     for i in range(len(x) - 1):
#         plt.plot(x[i:i+2], y[i:i+2], color=colors[i], linewidth=1)

# sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
# sm.set_array([])
# plt.colorbar(sm, ax=plt.gca())

# set_labels(r"$k_x$", r"$k_y$", r"Gap (meV)")

# save_plot("gap_contour.png")

# for i,(x,y,z) in enumerate(xyzs):
#     theta = np.arctan2(y,x)
#     idx = np.argsort(theta)
#     plt.plot(theta[idx], z[idx], linewidth=1)
#     set_labels(r"$\theta$", r"Gap (meV)")
#     plt.xticks(np.linspace(-np.pi, np.pi, 5), [r"$-\pi$", r"$-\pi/2$", r"$0$", r"$\pi/2$", r"$\pi$"])
#     save_plot(f"gap_contour_{i}.png")


# for ax in axes.flat:
# ax.set_xlabel(r"$\theta$")
# ax.set_ylabel(r"Gap (meV)")
# ax.set_yscale("log")
# ax.set_xticks(np.linspace(-np.pi, np.pi, 5))
# ax.set_xticklabels([r"$-\pi$", r"$-\pi/2$", r"$0$", r"$\pi/2$", r"$\pi$"])
# ax.set_ylim(1e-1, 1e-0)
# ax.set_ylim(1e-1, 1e-0)
#     ax.label_outer()
# fig.tight_layout()
# fig.savefig("plot/gap_contour.png", )

### testing

In [22]:
kx, ky, *Es = load_data("test.dat")

n_bands = 2

fig, axes = plt.subplots(2, n_bands//2, sharex=True, sharey=True, figsize=(7, 5))
for i, ax in enumerate(axes.flat):
    Z, e = prep_image_data(kx, ky, Es[i]+Es[i+n_bands])

    mapped = ax.imshow(
        symlog10(Z), extent=e, origin="lower", aspect="equal", cmap="RdBu"
    )
    
    ax.set_xlabel(r"$k_x$")
    ax.set_ylabel(r"$k_y$")
    ax.label_outer()
    ax.set_box_aspect(1)
    fig.colorbar(mapped, ax=ax)

save_plot(f"test.png", dpi=600)

### test2

In [23]:
kx, ky, *absDeltas = load_data("test2.dat")

n_bands = 2

fig, axes = plt.subplots(n_bands, n_bands, sharex=True, sharey=True, figsize=(10, 10))
for i, ax in enumerate(axes.flat):
    Z, e = prep_image_data(kx, ky, absDeltas[i])

    mapped = ax.imshow(
        Z, extent=e, origin="lower", aspect="equal", cmap="Reds", 
    )
    
    ax.set_xlabel(r"$k_x$")
    ax.set_ylabel(r"$k_y$")
    ax.label_outer()
    ax.set_box_aspect(1)
    fig.colorbar(mapped, ax=ax)

save_plot(f"test2.png", dpi=600)

### test 3

In [24]:
kx, ky, *traces = load_data("test3.dat")

n_bands = 2

fig, axes = plt.subplots(n_bands//2, 2, sharex=True, sharey=True, figsize=(5, 7))
for i, ax in enumerate(axes.flat):
    Z, e = prep_image_data(kx, ky, np.sqrt(traces[i]/2))

    mapped = ax.imshow(
        Z, extent=e, origin="lower", aspect="equal", cmap="RdBu"
    )
    print(np.min(Z), np.max(Z))
    ax.set_xlabel(r"$k_x$")
    ax.set_ylabel(r"$k_y$")
    ax.label_outer()
    ax.set_box_aspect(1)
    fig.colorbar(mapped, ax=ax)

save_plot(f"test3.png", dpi=600)

0.2 126.9100074856195
0.0 0.0


### test 4

In [28]:
kx, ky, *traces = load_data("test4.dat")

fig, axes = plt.subplots(1, 2, sharex=True, sharey=True, figsize=(5, 3))
for i, ax in enumerate(axes.flat):
    Z, e = prep_image_data(kx, ky, traces[i])

    mapped = ax.imshow(
        Z, extent=e, origin="lower", aspect="equal", cmap="RdBu"
    )
    print(np.min(Z), np.max(Z))
    ax.set_xlabel(r"$k_x$")
    ax.set_ylabel(r"$k_y$")
    ax.label_outer()
    ax.set_box_aspect(1)
    fig.colorbar(mapped, ax=ax)

save_plot(f"test4.png", dpi=600)

FileNotFoundError: build/data/test4.dat not found.

### numerical zeros of determinant

In [5]:
lam, *ds = load_data("det.dat")

dy = 0.002

fig, axes = plt.subplots(2,1, sharex=True, figsize=(4, 4))

for i, ax in enumerate(axes.flat):
    ax.plot(lam, ds[i])
    ax.set_xlabel(r"$\lambda$")
    ax.set_ylabel(r"$\det$")
    ax.grid()
    ax.set_ylim(-dy, dy)
    ax.axhline(0, color="red", linewidth=0.5, linestyle="--")
    
save_plot(f"det.png")